# Generate XMI from SSOT Model

In [1]:
source = '/Users/bue/dev/geberit/DEAP/DB/IM_GEBERIT.json'

In [2]:
from lxml import etree
from io import BytesIO

In [3]:
import json

In [4]:
with open(source, 'r') as src:
    ssot_model = json.load(src)

In [5]:
print(f"{ssot_model['_imprint_']}")

{'database': '/Users/bue/dev/geberit/DEAP/DB/IM_GEBERIT.db', 'created': '2021-11-10 22:46:23.021508', 'Modelversion': '1.5', 'hashvalue': 696266673081852930, 'comment': 'Entries ending with + represent denormalized data and are not checked for consistency while reading back', 'json': '/Users/bue/dev/geberit/DEAP/DB/IM_GEBERIT.json', 'model-git-revision': '72d5de1'}


In [6]:
root_frame = """<?xml version="1.0" encoding="windows-1252"?>
<xmi:XMI xmi:version="2.1" xmlns:uml="http://schema.omg.org/spec/UML/2.1" xmlns:xmi="http://schema.omg.org/spec/XMI/2.1" xmlns:thecustomprofile="http://www.sparxsystems.com/profiles/thecustomprofile/1.0" xmlns:Informationmodel="http://www.sparxsystems.com/profiles/Informationmodel/3.4">
	<xmi:Documentation exporter="Enterprise Architect" exporterVersion="6.5" exporterID="1559"/>
	<uml:Model xmi:type="uml:Model" name="EA_Model" visibility="public">
    </uml:Model>
</xmi:XMI>"""

In [7]:
parser = etree.XMLParser(remove_blank_text=True)
xmi_model = etree.fromstring(root_frame.encode('UTF-8'))

In [8]:
xmi_model.tag

'{http://schema.omg.org/spec/XMI/2.1}XMI'

In [9]:
assert len(list(xmi_model)), f"Check model load"

In [10]:
model_node = xmi_model.find('uml:Model', xmi_model.nsmap)

In [11]:
model_node

<Element {http://schema.omg.org/spec/UML/2.1}Model at 0x7fd849710740>

In [12]:
im_namespace = 'Informationmodel'
etree.register_namespace(im_namespace, 'Informationmodel')

# Base model is ready, now insert XMI

In [13]:
xmi_ns = '{' + model_node.nsmap['xmi'] + '}'

In [14]:
from uuid import uuid1

class IdentityMap(object):
    
    def __init__(self, prefix: str):
        self.map = dict()
        self.prefix = prefix

    def create(self, name: str) -> str:
        new_identity = uuid1()
        key = self.prefix + str(new_identity).upper().replace('-', '_')
        self.map[name] = key
        return key
    
    def create_mark(self, name: str, mark: str) -> str:
        new_identity = uuid1()
        key = self.prefix + str(new_identity).upper().replace('-', '_')
        key_marked = key[0:4] + '_' + mark + key[(4+len(mark)):]
        self.map[name] = key_marked
        return key_marked
        
    def get(self, name: str) -> str:
        return self.map[name]

In [15]:
package_map = IdentityMap('EAPK_')
class_map = IdentityMap('EAID_')
relation_map = IdentityMap('EAID_')

## Package to isolate new elements

```<packagedElement xmi:type="uml:Package" xmi:id="EAPK_B14E1732_D846_40e2_B810_8C739257682B" name="Model" visibility="public">```

In [16]:
key = package_map.create('Import')
package_node = etree.Element('packagedElement')
package_node.set(xmi_ns + 'type', 'uml:Package')
package_node.set(xmi_ns + 'id', key)
package_node.set('name', 'Import')
package_node.set('visibility', 'public')
model_node.insert(0, package_node)

### Add elements to package

#### Classes

```
<packagedElement xmi:type="uml:Class" xmi:id="EAID_1FDCDC17_0587_4f38_AFB1_3ED0159DA4A7" name="Airport" visibility="public">
					<ownedAttribute xmi:type="uml:Property" xmi:id="EAID_CB0F96CF_C29B_4f9a_8B72_FFE959450E46" name="IATA Code" visibility="public" isStatic="false" isReadOnly="false" isDerived="false" isOrdered="false" isUnique="true" isDerivedUnion="false">
						<type xmi:idref="EAID_7320D192_A2DF_463e_9FF6_A1E3A7987E33"/>
						<lowerValue xmi:type="uml:LiteralInteger" xmi:id="EAID_LI000001_C29B_4f9a_8B72_FFE959450E46" value="1"/>
						<upperValue xmi:type="uml:LiteralInteger" xmi:id="EAID_LI000002_C29B_4f9a_8B72_FFE959450E46" value="1"/>
					</ownedAttribute>
				</packagedElement>
```

In [17]:
parking_space_key = class_map.create('Parkingspace')
class_node = etree.Element('packagedElement')
class_node.set(xmi_ns + 'type', 'uml:Class')
class_node.set(xmi_ns + 'id', parking_space_key)
class_node.set('name', 'Parking Space')
class_node.set('visibility', 'public')
package_node.append(class_node)

im_namespace = 'Informationmodel'
etree.register_namespace(im_namespace, 'Informationmodel')

class_extension = etree.Element(etree.QName(im_namespace, 'Entity'))
class_extension.set('base_Class', parking_space_key)
model_node.append(class_extension)

In [18]:
car_key = class_map.create('Car')
car_node = etree.Element('packagedElement')
car_node.set(xmi_ns + 'type', 'uml:Class')
car_node.set(xmi_ns + 'id', car_key)
car_node.set('name', 'Car')
car_node.set('visibility', 'public')
package_node.append(car_node)

class_extension = etree.Element(etree.QName(im_namespace, 'Entity'))
class_extension.set('base_Class', car_key)
model_node.append(class_extension)

In [19]:
etree.tostring(model_node)

b'<uml:Model xmlns:uml="http://schema.omg.org/spec/UML/2.1" xmlns:xmi="http://schema.omg.org/spec/XMI/2.1" xmlns:thecustomprofile="http://www.sparxsystems.com/profiles/thecustomprofile/1.0" xmlns:Informationmodel="http://www.sparxsystems.com/profiles/Informationmodel/3.4" xmi:type="uml:Model" name="EA_Model" visibility="public">\n    <packagedElement xmi:type="uml:Package" xmi:id="EAPK_05A41C5C_5739_11EC_B456_1E00E2227870" name="Import" visibility="public"><packagedElement xmi:type="uml:Class" xmi:id="EAID_05A4BF0E_5739_11EC_B456_1E00E2227870" name="Parking Space" visibility="public"/><packagedElement xmi:type="uml:Class" xmi:id="EAID_05A56292_5739_11EC_B456_1E00E2227870" name="Car" visibility="public"/></packagedElement><Informationmodel:Entity xmlns:Informationmodel="Informationmodel" base_Class="EAID_05A4BF0E_5739_11EC_B456_1E00E2227870"/><Informationmodel:Entity xmlns:Informationmodel="Informationmodel" base_Class="EAID_05A56292_5739_11EC_B456_1E00E2227870"/></uml:Model>\n</xmi:XMI

### Relations

Example:
```
<packagedElement xmi:type="uml:Association" xmi:id="EAID_CF26F881_7A4A_4356_B7FB_2722EAB3ACBA" visibility="public">
					<memberEnd xmi:idref="EAID_dst26F881_7A4A_4356_B7FB_2722EAB3ACBA"/>
					<memberEnd xmi:idref="EAID_src26F881_7A4A_4356_B7FB_2722EAB3ACBA"/>
					<ownedEnd xmi:type="uml:Property" xmi:id="EAID_src26F881_7A4A_4356_B7FB_2722EAB3ACBA" visibility="public" association="EAID_CF26F881_7A4A_4356_B7FB_2722EAB3ACBA" isStatic="false" isReadOnly="false" isDerived="false" isOrdered="false" isUnique="true" isDerivedUnion="false" aggregation="none">
						<type xmi:idref="EAID_FEAE01B6_71E0_4663_B202_EB07FF57D2D5"/>
					</ownedEnd>
					<ownedEnd xmi:type="uml:Property" xmi:id="EAID_dst26F881_7A4A_4356_B7FB_2722EAB3ACBA" visibility="public" association="EAID_CF26F881_7A4A_4356_B7FB_2722EAB3ACBA" isStatic="false" isReadOnly="false" isDerived="false" isOrdered="false" isUnique="true" isDerivedUnion="false" aggregation="none">
						<type xmi:idref="EAID_326DF717_74E8_4f63_A720_A1A3CBD9AD26"/>
					</ownedEnd>
				</packagedElement>
```

In [20]:
relation_parent_key = relation_map.create('Parkingspace_Car_Association')
relation_parent_node = etree.Element('packagedElement')
relation_parent_node.set(xmi_ns + 'type', 'uml:Association')
relation_parent_node.set(xmi_ns + 'id', relation_parent_key)

# memberEnd
relation_src_key = class_map.create_mark('Parkingspace_Association_src', 'src')
print(f"Source end key: {relation_src_key}")
relation_src_ref = etree.Element('memberEnd')
relation_src_ref.set(xmi_ns + 'idref', relation_src_key)
relation_parent_node.append(relation_src_ref)

# memberEnd
relation_dst_key = class_map.create_mark('Parkingspace_Association_dst', 'dst')
print(f"Destination end key: {relation_dst_key}")
relation_dst_ref = etree.Element('memberEnd')
relation_dst_ref.set(xmi_ns + 'idref', relation_dst_key)
relation_parent_node.append(relation_dst_ref)

# ownedEnd src
relation_src = etree.Element('ownedEnd')
relation_src.set(xmi_ns + 'type', 'uml:Property')
relation_src.set(xmi_ns + 'id', relation_src_key)
relation_src.set('visiblity', 'public')
relation_src.set('association', relation_parent_key)
relation_src.set('name', "source name")
relation_src.set('isStatic', "false")
relation_src.set('isReadOnly', "false")
relation_src.set('isDerived', "false")
relation_src.set('isOrdered', "false")
relation_src.set('isUnique', "true")
relation_src.set('isDerivedUnion', "false")
relation_src.set('aggregation', "none")

src_class_ref = etree.Element('type')
src_class_ref.set(xmi_ns + 'idref', car_key)
relation_src.append(src_class_ref)

relation_parent_node.append(relation_src)

# ownedEnd src
relation_dst = etree.Element('ownedEnd')
relation_dst.set(xmi_ns + 'type', 'uml:Property')
relation_dst.set(xmi_ns + 'id', relation_dst_key)
relation_dst.set('visiblity', 'public')
relation_dst.set('association', relation_parent_key)
relation_dst.set('name', "destination name")
relation_dst.set('isStatic', "false")
relation_dst.set('isReadOnly', "false")
relation_dst.set('isDerived', "false")
relation_dst.set('isOrdered', "false")
relation_dst.set('isUnique', "true")
relation_dst.set('isDerivedUnion', "false")
relation_dst.set('aggregation', "none")
relation_parent_node.append(relation_dst)

# cardinalities

dst_class_ref = etree.Element('type')
dst_class_ref.set(xmi_ns + 'idref', parking_space_key)
relation_dst.append(dst_class_ref)

package_node.append(relation_parent_node)

#	<Informationmodel:Relation base_Association="EAID_21232F72_243C_48df_ADC2_E54CCBF71773"/>
class_extension = etree.Element(etree.QName(im_namespace, 'Relation'))
class_extension.set('base_Association', relation_parent_key)
model_node.append(class_extension)

Source end key: EAID_srcA73496_5739_11EC_B456_1E00E2227870
Destination end key: EAID_dstA744B8_5739_11EC_B456_1E00E2227870


In [21]:
etree.tostring(xmi_model)

b'<xmi:XMI xmlns:uml="http://schema.omg.org/spec/UML/2.1" xmlns:xmi="http://schema.omg.org/spec/XMI/2.1" xmlns:thecustomprofile="http://www.sparxsystems.com/profiles/thecustomprofile/1.0" xmlns:Informationmodel="http://www.sparxsystems.com/profiles/Informationmodel/3.4" xmi:version="2.1">\n\t<xmi:Documentation exporter="Enterprise Architect" exporterVersion="6.5" exporterID="1559"/>\n\t<uml:Model xmi:type="uml:Model" name="EA_Model" visibility="public">\n    <packagedElement xmi:type="uml:Package" xmi:id="EAPK_05A41C5C_5739_11EC_B456_1E00E2227870" name="Import" visibility="public"><packagedElement xmi:type="uml:Class" xmi:id="EAID_05A4BF0E_5739_11EC_B456_1E00E2227870" name="Parking Space" visibility="public"/><packagedElement xmi:type="uml:Class" xmi:id="EAID_05A56292_5739_11EC_B456_1E00E2227870" name="Car" visibility="public"/><packagedElement xmi:type="uml:Association" xmi:id="EAID_05A72F32_5739_11EC_B456_1E00E2227870"><memberEnd xmi:idref="EAID_srcA73496_5739_11EC_B456_1E00E2227870"

In [22]:
destination = 'test-2.1.xmi'
et = etree.ElementTree(xmi_model)
et.write(destination, pretty_print=True)